# Governing LangGraph Tool Calls with Talon

This notebook demonstrates the actual integration path:

```text
LangGraph agent -> ChatOpenAI client -> Talon OpenAI-compatible gateway -> OpenAI
```

LangGraph uses the normal LangChain `ChatOpenAI` client. The only integration change is:

```python
base_url="http://localhost:18080/v1/proxy/openai/v1"
api_key=CALLER_KEY
```

This makes Talon the governance boundary for LangGraph model requests and OpenAI-compatible tool definitions.

The notebook validates:

- LangGraph runs through Talon as a proxy.
- Talon allows safe tool definitions.
- Talon filters dangerous tool definitions before OpenAI sees them.
- Talon blocks the whole request in strict mode.
- Talon enforces a model allowlist.
- Talon writes HMAC-signed evidence.
- Talon audit/export/report/metrics can be used for governance reporting.


## 0. Setup

Use port `18080` to avoid Colab conflicts. The notebook stops only the Talon PID it starts.

In [ ]:
import os, json, time, signal, subprocess, re, zipfile, shutil
from pathlib import Path
from getpass import getpass

PROJECT = Path("/content/talon-langgraph-tool-governance")
PROJECT.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT / ".talon"
DATA_DIR.mkdir(exist_ok=True)

PORT = 18080
ADMIN_KEY = "talon-admin-demo"
CALLER_KEY = "talon-gw-langgraph-tools-demo"
AGENT_NAME = "langgraph-tool-agent"
GATEWAY_TENANT = "demo"  # must match gateway.callers[].tenant_id in talon.config.yaml

os.environ["TALON_DATA_DIR"] = str(DATA_DIR)
os.environ["TALON_ADMIN_KEY"] = ADMIN_KEY
os.environ["TALON_SECRETS_KEY"] = "0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef"
os.environ["TALON_SIGNING_KEY"] = "abcdef0123456789abcdef0123456789abcdef0123456789abcdef0123456789"

def run(cmd, cwd=None, env=None, check=True, capture=True):
    printable = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    print("$", printable)
    p = subprocess.run(cmd, cwd=cwd, env=env or os.environ.copy(), shell=isinstance(cmd, str), text=True, capture_output=capture)
    if capture:
        if p.stdout: print(p.stdout)
        if p.stderr: print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {printable}")
    return p.stdout if capture else ""

print("Project:", PROJECT)

## 1. Install Talon

The cell tries: existing binary, attached artifact, official installer, then Go fallback.

In [ ]:
def find_talon():
    for candidate in ["/usr/local/bin/talon", "/content/talon", "/tmp/talon-bin/talon", str(Path.home()/"go/bin/talon")]:
        p = Path(candidate)
        if p.exists():
            p.chmod(0o755)
            return str(p)
    return shutil.which("talon")

def install_from_attached_artifact():
    zip_path = Path("/mnt/data/talon-linux-amd64.zip")
    if not zip_path.exists():
        return None
    out = Path("/tmp/talon-bin")
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out)
    for candidate in [out/"talon", *out.glob("**/talon")]:
        if candidate.exists():
            candidate.chmod(0o755)
            return str(candidate)
    return None

def install_talon():
    talon_path = find_talon()
    if talon_path:
        print("Talon already available:", talon_path)
        return talon_path
    artifact = install_from_attached_artifact()
    if artifact:
        print("Using attached Talon artifact:", artifact)
        return artifact
    print("Installing Talon via official installer...")
    p = subprocess.run("curl -fsSL https://install.gettalon.dev | INSTALL_DIR=/usr/local/bin sh", shell=True, text=True, capture_output=True)
    print(p.stdout); print(p.stderr)
    talon_path = find_talon()
    if talon_path: return talon_path
    print("Installing modern Go fallback...")
    run("curl -fsSL https://go.dev/dl/go1.23.4.linux-amd64.tar.gz -o /tmp/go.tar.gz")
    run("rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tar.gz")
    env = os.environ.copy()
    env["PATH"] = "/usr/local/go/bin:" + env.get("PATH", "")
    env["GONOSUMDB"] = "github.com/dativo-io/talon"
    env["GOPROXY"] = "direct"
    run("go version", env=env)
    run("go install github.com/dativo-io/talon/cmd/talon@latest", env=env)
    os.environ["PATH"] = env["PATH"] + ":" + str(Path.home()/"go/bin")
    talon_path = find_talon()
    if not talon_path:
        raise RuntimeError("Talon installation failed")
    return talon_path

TALON = install_talon()
print("Using Talon binary:", TALON)
run([TALON, "version"], check=False)

## 2. Load OpenAI key and store it in Talon's vault

LangGraph will use the Talon caller key. The real provider key stays in Talon.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

openai_key = os.environ["OPENAI_API_KEY"]

## 3. Create Talon config

`tool_policy_action` is switched between `filter` and `block` in later examples.

In [ ]:
agent_policy = f"""
agent:
  name: {AGENT_NAME}
  version: "1.0.0"
  description: Real LangGraph-through-Talon tool-governance demo

capabilities:
  allowed_tools: []

policies:
  data_classification:
    input_scan: true
    output_scan: true
    redact_pii: true
    block_on_pii: false
  cost_limits:
    per_request: 0.50
    daily: 10.00
    monthly: 200.00

audit:
  log_level: detailed
  retention_days: 30
  log_prompts: false
  log_responses: false

compliance:
  frameworks:
    - gdpr
    - eu-ai-act
  data_residency: eu
  risk_level: low
"""
(PROJECT/"agent.talon.yaml").write_text(agent_policy)
print((PROJECT/"agent.talon.yaml").read_text())

def write_gateway_config(tool_policy_action="filter", allowed_models=None):
    allowed_models = allowed_models or ["gpt-4o-mini"]
    model_lines = "\n".join([f"          - \"{m}\"" for m in allowed_models])
    config = f"""
gateway:
  mode: enforce

  providers:
    openai:
      enabled: true
      base_url: "https://api.openai.com"
      secret_name: "openai-api-key"
      allowed_models:
{model_lines}

  default_policy:
    default_pii_action: "redact"
    response_pii_action: "warn"
    tool_policy_action: "{tool_policy_action}"
    forbidden_tools:
      - "export_*"
      - "delete_*"
      - "admin_*"
      - "drop_*"
      - "truncate_*"
      - "bulk_*"
    max_daily_cost: 10.00
    max_monthly_cost: 200.00
    allowed_models:
{model_lines}

  callers:
    - name: "{AGENT_NAME}"
      tenant_key: "{CALLER_KEY}"
      tenant_id: "{GATEWAY_TENANT}"
      allowed_providers:
        - "openai"
      policy_overrides:
        pii_action: "redact"
        response_pii_action: "warn"
        tool_policy_action: "{tool_policy_action}"
        allowed_tools:
          - "search_records"
          - "update_record"
          - "send_notification"
        forbidden_tools:
          - "export_*"
          - "delete_*"
          - "admin_*"
          - "drop_*"
          - "truncate_*"
          - "bulk_*"
        allowed_models:
{model_lines}
        max_daily_cost: 10.00
        max_monthly_cost: 200.00
"""
    (PROJECT/"talon.config.yaml").write_text(config)
    print(config)

write_gateway_config("filter")

## 4. Start Talon gateway

In [ ]:
TALON_PID = PROJECT/"talon.pid"
TALON_LOG = PROJECT/"talon.log"

def stop_talon():
    if TALON_PID.exists():
        pid = int(TALON_PID.read_text())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(1)
        except ProcessLookupError:
            pass
        TALON_PID.unlink(missing_ok=True)

def start_talon():
    stop_talon()
    env = os.environ.copy()
    env["TALON_DATA_DIR"] = str(DATA_DIR)
    env["TALON_ADMIN_KEY"] = ADMIN_KEY
    env["TALON_SECRETS_KEY"] = os.environ["TALON_SECRETS_KEY"]
    env["TALON_SIGNING_KEY"] = os.environ["TALON_SIGNING_KEY"]
    log = open(TALON_LOG, "w")
    p = subprocess.Popen([TALON, "serve", "--gateway", "--gateway-config", str(PROJECT/"talon.config.yaml"), "--port", str(PORT)], cwd=PROJECT, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
    TALON_PID.write_text(str(p.pid))
    for _ in range(60):
        status = subprocess.run(f"curl -s -o /dev/null -w '%{{http_code}}' http://localhost:{PORT}/v1/status", shell=True, text=True, capture_output=True).stdout.strip()
        if status in {"200", "401"}:
            print("Talon gateway is up on port", PORT)
            return
        if p.poll() is not None:
            raise RuntimeError("Talon exited early:\n" + TALON_LOG.read_text())
        time.sleep(1)
    raise RuntimeError("Talon did not become ready:\n" + TALON_LOG.read_text())

start_talon()
print(TALON_LOG.read_text()[-2000:])

## 5. Install LangGraph dependencies

In [ ]:
run("pip install -q langgraph langchain-openai langchain-core")

## 6. Define LangGraph tools

Safe tools: `search_records`, `update_record`, `send_notification`.

Dangerous tools: `export_data`, `delete_record`, `admin_override`.

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

@tool
def search_records(query: str) -> str:
    """Search business records by query."""
    return f"Found records matching: {query}"

@tool
def update_record(record_id: str, status: str) -> str:
    """Update a single business record."""
    return f"Updated {record_id} to status={status}"

@tool
def send_notification(recipient: str, message: str) -> str:
    """Send a notification to a recipient."""
    return f"Notification sent to {recipient}: {message}"

@tool
def export_data(scope: str) -> str:
    """Export company data for a given scope."""
    return f"Exported data for scope={scope}"

@tool
def delete_record(record_id: str) -> str:
    """Delete a business record."""
    return f"Deleted record {record_id}"

@tool
def admin_override(reason: str) -> str:
    """Override policy controls."""
    return f"Admin override executed: {reason}"

SAFE_TOOLS = [search_records, update_record, send_notification]
DANGEROUS_TOOLS = [export_data, delete_record, admin_override]
MIXED_TOOLS = SAFE_TOOLS + DANGEROUS_TOOLS
print("Safe tools:", [t.name for t in SAFE_TOOLS])
print("Dangerous tools:", [t.name for t in DANGEROUS_TOOLS])

## 7. Create `ChatOpenAI` through Talon

This is the proxy integration. LangGraph calls LangChain, LangChain calls Talon, Talon calls OpenAI.

In [ ]:
def make_llm(model="gpt-4o-mini"):
    return ChatOpenAI(
        model=model,
        base_url=f"http://localhost:{PORT}/v1/proxy/openai/v1",
        api_key=CALLER_KEY,
        temperature=0,
        max_tokens=120,
    )

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def build_graph(bound_llm):
    def support_agent(state: AgentState):
        response = bound_llm.invoke(state["messages"])
        return {"messages": [response]}
    builder = StateGraph(AgentState)
    builder.add_node("support_agent", support_agent)
    builder.add_edge(START, "support_agent")
    builder.add_edge("support_agent", END)
    return builder.compile()

def run_langgraph_with_tools(tools, prompt, model="gpt-4o-mini"):
    llm = make_llm(model=model).bind_tools(tools)
    graph = build_graph(llm)
    return graph.invoke({"messages": [
        SystemMessage(content="You are a careful business operations assistant. Do not execute destructive actions."),
        HumanMessage(content=prompt),
    ]})

print("LangGraph helper ready. Talon proxy URL:", f"http://localhost:{PORT}/v1/proxy/openai/v1")

## 8. Evidence helpers

Assertions are whitespace-tolerant because `talon audit show` is human-readable.

In [ ]:
def talon_cli(args, check=True):
    return run([TALON] + args, cwd=PROJECT, check=check)

def latest_evidence_id(agent=AGENT_NAME):
    text = talon_cli(["audit", "list", "--agent", agent, "--limit", "1"], check=False)
    ids = re.findall(r"\b(?:gw|req|evt)_[A-Za-z0-9_-]+\b", text)
    if not ids:
        raise RuntimeError("No evidence ID found in audit list:\n" + text)
    return ids[0]

def show_latest(agent=AGENT_NAME):
    eid = latest_evidence_id(agent)
    print("Latest evidence:", eid)
    show = talon_cli(["audit", "show", eid], check=False)
    verify = talon_cli(["audit", "verify", eid], check=False)
    assert "VALID" in verify.upper(), verify
    return eid, show, verify

def evidence_has_field(text, field, value):
    pattern = rf"(?im)^\s*{re.escape(field)}:\s*{re.escape(str(value))}\s*$"
    return re.search(pattern, text) is not None

def assert_evidence_status(text, allowed=None, action=None, reason=None):
    if allowed is not None:
        assert evidence_has_field(text, "Allowed", str(allowed).lower()), text
    if action is not None:
        assert evidence_has_field(text, "Action", action), text
    if reason is not None:
        assert re.search(re.escape(reason), text, re.IGNORECASE), text

def assert_tool_governance(show, requested=None, filtered=None, forwarded=None):
    for name in requested or []:
        assert name in show, f"{name} missing from evidence.\n{show}"
    for name in filtered or []:
        assert name in show, f"{name} missing from filtered list.\n{show}"
    for name in forwarded or []:
        assert name in show, f"{name} missing from forwarded list.\n{show}"

print("Evidence helpers ready.")

## 9. Example A — LangGraph with allowed tools only

Expected: allowed, no filtered tools, signed evidence.

In [ ]:
write_gateway_config("filter")
start_talon()

result_allowed = run_langgraph_with_tools(SAFE_TOOLS, "Find records matching Project Phoenix and notify owners. Do not delete anything.")
print(result_allowed["messages"][-1])

eid_allowed, show_allowed, _ = show_latest()
assert_evidence_status(show_allowed, allowed=True, action="allow")
assert_tool_governance(show_allowed, requested=["search_records", "update_record", "send_notification"], forwarded=["search_records", "update_record", "send_notification"])
assert "Filtered:" in show_allowed and "(none)" in show_allowed
print("PASSED LangGraph allowed-only case:", eid_allowed)

## 10. Example B — LangGraph with mixed tools in `filter` mode

Expected: dangerous tools are filtered, safe tools are forwarded, request is allowed.

In [ ]:
write_gateway_config("filter")
start_talon()

result_filter = run_langgraph_with_tools(MIXED_TOOLS, "Find records matching Project Phoenix and notify owners. Do not delete or export anything.")
print(result_filter["messages"][-1])

eid_filter, show_filter, _ = show_latest()
assert_evidence_status(show_filter, allowed=True, action="allow")
assert_tool_governance(
    show_filter,
    requested=["search_records", "update_record", "send_notification", "export_data", "delete_record", "admin_override"],
    filtered=["export_data", "delete_record", "admin_override"],
    forwarded=["search_records", "update_record", "send_notification"],
)
print("PASSED LangGraph filter-mode mixed tools case:", eid_filter)

## 11. Example C — LangGraph with dangerous tools in `block` mode

Expected: Talon denies before OpenAI call; evidence is still written and valid.

In [ ]:
write_gateway_config("block")
start_talon()

try:
    result_block = run_langgraph_with_tools(MIXED_TOOLS, "Export all company records and delete the originals.")
    print(result_block["messages"][-1])
except Exception as e:
    print("LangGraph call failed as expected because Talon blocked the request:")
    print(type(e).__name__, str(e)[:1000])

eid_block, show_block, _ = show_latest()
assert_evidence_status(show_block, allowed=False, action="deny", reason="tool governance block")
assert "POLICY_DENIED_TOOL" in show_block
assert_tool_governance(show_block, filtered=["export_data", "delete_record", "admin_override"])
print("PASSED LangGraph block-mode case:", eid_block)

## 12. Example D — model allowlist denial through LangGraph

Expected: LangGraph asks for `gpt-4o`, Talon denies because only `gpt-4o-mini` is allowed. Evidence should use explanation code `POLICY_DENIED_ROUTING` on current Talon builds.

In [ ]:
write_gateway_config("filter", allowed_models=["gpt-4o-mini"])
start_talon()

try:
    result_model = run_langgraph_with_tools(SAFE_TOOLS, "Say hello.", model="gpt-4o")
    print(result_model["messages"][-1])
except Exception as e:
    print("LangGraph call failed as expected because Talon blocked the disallowed model:")
    print(type(e).__name__, str(e)[:1000])

eid_model, show_model, _ = show_latest()
assert_evidence_status(show_model, allowed=False, action="deny")
assert re.search(r"model\s+gpt-4o\s+not\s+in\s+caller\s+allowlist", show_model, re.IGNORECASE), show_model
assert (
    "POLICY_DENIED_ROUTING" in show_model
    or "LEGACY_REASON_UNMIGRATED" in show_model
), "expected POLICY_DENIED_ROUTING (Talon main) or legacy explanation code"
print("PASSED LangGraph model allowlist denial:", eid_model)

## 13. Advanced reporting: audit list/show/verify/export

In [ ]:
print("Recent evidence:")
talon_cli(["audit", "list", "--agent", AGENT_NAME, "--limit", "10"], check=False)

print("\nExport JSON:")
json_export = talon_cli(["audit", "export", "--format", "json", "--limit", "20"], check=False)
(PROJECT/"talon-langgraph-tool-governance-evidence.json").write_text(json_export)
print("Saved:", PROJECT/"talon-langgraph-tool-governance-evidence.json")

print("\nExport CSV:")
csv_export = talon_cli(["audit", "export", "--format", "csv", "--limit", "20"], check=False)
(PROJECT/"talon-langgraph-tool-governance-evidence.csv").write_text(csv_export)
print("Saved:", PROJECT/"talon-langgraph-tool-governance-evidence.csv")

for eid in [eid_allowed, eid_filter, eid_block, eid_model]:
    verify = talon_cli(["audit", "verify", eid], check=False)
    assert "VALID" in verify.upper(), verify
print("All captured evidence records verify successfully.")

## 14. Advanced reporting: report, costs, metrics, dashboard

In [ ]:
print(f"talon report (tenant {GATEWAY_TENANT}):")
talon_cli(["report", "--tenant", GATEWAY_TENANT], check=False)

print(f"\ntalon costs (tenant {GATEWAY_TENANT}):")
talon_cli(["costs", "--tenant", GATEWAY_TENANT], check=False)

import requests
metrics_url = f"http://localhost:{PORT}/api/v1/metrics"
m = requests.get(metrics_url, headers={"X-Talon-Admin-Key": ADMIN_KEY}, timeout=20)
print("Metrics HTTP", m.status_code)
try:
    metrics = m.json()
    print(json.dumps(metrics, indent=2)[:5000])
    summary = metrics.get("summary", {})
    print("tools_filtered:", summary.get("tools_filtered"))
    print("blocked_requests:", summary.get("blocked_requests"))
    print("tool_governance:", json.dumps(metrics.get("tool_governance", {}), indent=2))
except Exception:
    print(m.text[:5000])

print("\nDashboard URL:")
print(f"http://localhost:{PORT}/gateway/dashboard?talon_admin_key={ADMIN_KEY}")

## 15. Optional diagnostic: raw OpenAI-compatible request

The main demo uses LangGraph. Keep raw HTTP only for debugging.

In [ ]:
# Optional diagnostic only. Uncomment if needed.
# import requests
# payload = {
#     "model": "gpt-4o-mini",
#     "messages": [{"role": "user", "content": "Find Project Phoenix records."}],
#     "tools": [],
#     "max_tokens": 50,
# }
# r = requests.post(
#     f"http://localhost:{PORT}/v1/proxy/openai/v1/chat/completions",
#     headers={"Authorization": f"Bearer {CALLER_KEY}", "Content-Type": "application/json"},
#     json=payload,
# )
# print(r.status_code, r.text[:1000])

## 16. Cleanup

In [ ]:
stop_talon()
print("Stopped Talon.")

# What this notebook proved

The tested path is:

```text
LangGraph -> LangChain ChatOpenAI -> Talon Gateway -> OpenAI
```

The critical application change is:

```python
llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="http://localhost:18080/v1/proxy/openai/v1",
    api_key="talon-gw-langgraph-tools-demo",
)
```

Talon then governs tool definitions and model access before the request reaches OpenAI.

Blog thesis:

> Prompting is not governance. A dangerous tool should be removed or blocked before the model can use it.
